# Medical Insurance Cost Risk — Prediction Demo

This notebook illustrates a **single example prediction** using the classifier selected by **training-only cross-validation** in `reports/metrics.json`. To keep it reproducible and safe, we fit the selected model on the training split rather than loading an untrusted `.joblib` file. The demonstration is not a real underwriting recommendation or clinical conclusion.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = next((p for p in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent) if (p / "src" / "pipeline.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Project source folder not found; start Jupyter from repository or notebooks/ folder")
if str(ROOT.resolve()) not in sys.path:
    sys.path.insert(0,str(ROOT.resolve()))

from src.pipeline import engineer, make_target, SEED, clean_training_data
from src.data import ensure_dataset
from src.train import build_models
from src.predict import predict_from_rows

data_file = ensure_dataset(ROOT / "data" / "insurance.csv")
raw, duplicates_removed = clean_training_data(pd.read_csv(data_file))
X = engineer(raw)
y = make_target(raw["charges"])
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=SEED,stratify=y)
results = json.loads((ROOT / "reports" / "metrics.json").read_text())
selected_name = results["selected_model"]
model = build_models()[selected_name].fit(X_train,y_train)
example = {"age":35,"sex":"female","bmi":27.1,"children":1,"smoker":"no","region":"northeast"}
print("Selected classifier:", selected_name)
print("Illustrative prediction:", predict_from_rows(model,[example]))

Selected classifier: CatBoost
Illustrative prediction: [{'high_cost_probability': 0.0604, 'high_cost_prediction_at_0_5': 0}]
